<a href="https://colab.research.google.com/github/GaPau/MooCraDee/blob/add-colab-gpu-demo/notebooks/PCS_GPU_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PCS GPU Demo

This notebook provides a reproducible GPU-enabled workflow for running Planetary Crater Segmentation (PCS), a crater-candidate detection pipeline for Mercury images.

PCS explores how foundation-model segmentation and computer vision can support the creation of a reproducible crater-candidate database.


## 1. GPU Setup


In [10]:
import torch

if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Go to Runtime > Change runtime type > GPU")

GPU available: Tesla T4


## 2. Install Dependencies

In [11]:
!pip install opencv-python pillow matplotlib numpy torch torchvision
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-c9ae0209
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-c9ae0209
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done


## 3. Clone Repository and Set Working Directory


In [8]:
!git clone https://github.com/GaPau/MooCraDee.git
%cd MooCraDee

Cloning into 'MooCraDee'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 188 (delta 8), reused 1 (delta 1), pack-reused 165 (from 3)
Receiving objects: 100% (188/188), 511.00 MiB | 45.70 MiB/s, done.
Resolving deltas: 100% (35/35), done.
Updating files: 100% (57/57), done.
/content/MooCraDee/MooCraDee


In [9]:
!git checkout add-colab-gpu-demo

Branch 'add-colab-gpu-demo' set up to track remote branch 'add-colab-gpu-demo' from 'origin'.
Switched to a new branch 'add-colab-gpu-demo'


## 4. Download SAM Checkpoint

In [13]:
from pathlib import Path

ckpt_path = Path("sam_vit_b_01ec64.pth")

if not ckpt_path.exists():
    print("Downloading SAM ViT-B checkpoint...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print("Download complete.")
else:
    print("SAM checkpoint already exists.")

Download complete.


## 5. Verify Required Files

In [14]:
from pathlib import Path

required_files = [
    "mercury.jpg",
    "run_pipeline.py",
    "deep_moocrade.py",
    "split_image.py",
    "sam_vit_b_01ec64.pth"
]

for file in required_files:
    if Path(file).exists():
        print(f"Found: {file}")
    else:
        print(f"Missing: {file}")

Found: mercury.jpg
Found: run_pipeline.py
Found: deep_moocrade.py
Found: split_image.py
Found: sam_vit_b_01ec64.pth


## 4. Select Input Image


In [15]:
!python run_pipeline.py 6

Guardado: mercury_split_6/part_1/input.png
Guardado: mercury_split_6/part_2/input.png
Guardado: mercury_split_6/part_3/input.png
Guardado: mercury_split_6/part_4/input.png
Guardado: mercury_split_6/part_5/input.png
Guardado: mercury_split_6/part_6/input.png

Done. Created 6 folders in: mercury_split_6

Running detector on part 1
Input: mercury_split_6/part_1/input.png
Output image: mercury_split_6/part_1/detected.png
CSV: mercury_split_6/part_1/radii.csv
Using GPU/CUDA: Tesla T4
Device: cuda
Circulos finales: 44
1: centro=(877.3,932.5)  radio=23.7px  score=1.893
2: centro=(1011.6,909.0)  radio=28.6px  score=1.886
3: centro=(1143.0,953.5)  radio=26.0px  score=1.883
4: centro=(838.0,374.0)  radio=21.1px  score=1.872
5: centro=(941.0,426.5)  radio=38.1px  score=1.871
6: centro=(832.1,600.0)  radio=36.2px  score=1.864
7: centro=(652.5,715.5)  radio=24.1px  score=1.863
8: centro=(1241.1,969.3)  radio=24.0px  score=1.857
9: centro=(541.7,762.9)  radio=21.7px  score=1.855
10: centro=(1149.5,7

Donde estas y que se creo, despues de la pipeline


In [16]:
!pwd
!ls
!ls mercury_split_6

/content/MooCraDee/MooCraDee
art2moon.jpg	  mercury.jpg	   __pycache__		 split_image.py
assets		  mercury_split_2  README.md
deep_moocrade.py  mercury_split_6  run_pipeline.py
examples	  notebooks	   sam_vit_b_01ec64.pth
all_craters.csv  part_1  part_2  part_3  part_4  part_5  part_6


## Run PCS on a Single Test Image

In [17]:
!python deep_moocrade.py art2moon.jpg \
  --ckpt sam_vit_b_01ec64.pth \
  --out art2moon_detected.png \
  --csv art2moon_craters.csv \
  --min_radius 20 \
  --max_radius 260 \
  --min_circularity 0.35 \
  --min_area 600 \
  --pps 64 \
  --pred_iou 0.80 \
  --stability 0.85 \
  --iou_dedup 0.12

Using GPU/CUDA: Tesla T4
Device: cuda
Circulos finales: 174
1: centro=(1647.2,185.4)  radio=31.0px  score=1.904
2: centro=(1597.5,82.0)  radio=22.6px  score=1.888
3: centro=(1244.0,42.0)  radio=28.2px  score=1.882
4: centro=(1343.8,217.6)  radio=24.0px  score=1.877
5: centro=(1866.5,171.5)  radio=29.8px  score=1.869
6: centro=(1621.0,479.5)  radio=30.3px  score=1.867
7: centro=(1216.0,175.5)  radio=80.7px  score=1.862
8: centro=(1016.0,307.0)  radio=38.8px  score=1.856
9: centro=(1591.5,1167.0)  radio=25.5px  score=1.855
10: centro=(1648.9,356.4)  radio=23.2px  score=1.855
11: centro=(1245.6,417.8)  radio=22.9px  score=1.855
12: centro=(1760.7,662.7)  radio=23.8px  score=1.842
13: centro=(1848.5,323.5)  radio=23.5px  score=1.841
14: centro=(1331.1,324.1)  radio=30.4px  score=1.840
15: centro=(1433.5,494.7)  radio=30.0px  score=1.838
16: centro=(1329.0,410.0)  radio=22.0px  score=1.836
17: centro=(697.6,378.4)  radio=22.8px  score=1.833
18: centro=(1537.0,543.5)  radio=24.4px  score=1.8

## 5. Set Detection Parameters

## 6. Run PCS Pipeline


## 7. Visualize Results

## 8. Export Crater Candidate Database

# New Section

In [ ]:
from google.colab import drive
drive.mount('/content/drive')